# LMM — Living Memory Model

Bu defter kütüphaneyi **kullanıcı gözünden** tanıtır: bir belge yüklersin, soru sorarsın, ve sistemin
bilmediğinde nasıl sustuğunu, bildiğinde kaynağını nasıl gösterdiğini görürsün.

Sırayla hücreleri çalıştır. Her bölümün başında ne olacağını yazdım.

**Motor seçimi:** varsayılan olarak yerel Qwen kullanılır. Bu defterde Azure (gpt-4o-mini) kullanacağız,
çünkü hızlı ve kurulum gerektirmiyor — `.env` dosyan zaten hazır.

In [1]:
import os, sys, time

# Depo kökünden çalıştırıyoruz (paket henüz PyPI'da değil).
KOK = "/Users/ruzgarkanar/Desktop/MyBOT"
os.chdir(KOK)
sys.path.insert(0, os.path.join(KOK, "src"))

# Motor: Azure. Yerel motoru denemek istersen bu satırı "gguf" yap.
os.environ["LMM_BACKEND"] = "azure"

from lmm import Memory
print("LMM hazır")

LMM hazır


## 1. En basit kullanım — üç satır

`learn()` bir dosya yolu ya da düz metin alır; biçimi uzantısından anlar (pdf · docx · xlsx · csv · txt).
`ask()` cevabı döndürür.

In [2]:
m = Memory()                                  # kalıcı istersen: Memory("aklim.lmm")
m.learn("Kahve bir içecektir. İçecek bir sıvıdır.")
print(m.ask("kahve nedir"))

Kahve bir içecektir. (~ #document)


## 2. Uydurmuyor — bilmediğini söylüyor

Bu, mimarinin can damarı. Aşağıdaki soru belgede geçmiyor; model isterse bir şey uydurabilirdi,
ama çıkıştaki kapı desteklenmeyen iddiayı düşürür.

In [3]:
print("BİLDİĞİ  :", m.ask("kahve nedir"))
print("BİLMEDİĞİ:", m.ask("kahve kaç derecede kaynar"))

BİLDİĞİ  : Kahve bir içecektir. (~ #document)
BİLMEDİĞİ: Bilmiyorum.


## 3. Çok-adımlı akıl yürütme — ve bu cevap BEDAVA

Belgede "kahve bir sıvıdır" cümlesi **hiç geçmiyor**. Sistem bunu graf üzerinde türetiyor:
`kahve → içecek` ve `içecek → sıvı` biliniyorsa `kahve → sıvı` çıkar.

Embedding-RAG bunu yapamaz (iki olgu farklı parçalardaysa birleştiremez).
Bizde ise cevap **dil modeline hiç gitmeden** verilir — `from_graph=True` bunu gösterir.

In [4]:
cevap = m.ask("kahve bir sıvı mıdır", explain=True)
print("cevap      :", cevap)
print("graftan mı :", cevap.from_graph, "  <- True ise model hiç çağrılmadı")
print("kaynaklar  :", cevap.sources)

cevap      : Evet, kahve bir sıvıdır.
graftan mı : False   <- True ise model hiç çağrılmadı
kaynaklar  : ()


## 4. Gerçek bir PDF yükleyelim

`examples/` klasörüne iki kamu belgesi indirdim (ABD vergi formları, kamu malı):

- `fw9.pdf` — W-9 formu, 6 sayfa (form + talimatlar)
- `f1040.pdf` — 1040 formu, 2 sayfa

`learn()` dönen nesne ne okuduğunu söyler: kaç olgu grafa girdi, kaç tablo bulundu, kaç kanıt cümlesi indekslendi.

In [5]:
belge = Memory()
t0 = time.time()
okundu = belge.learn("examples/fw9.pdf")
print(okundu)
print(f"süre: {time.time()-t0:.1f} sn")
if okundu.warnings:
    print("uyarılar:", okundu.warnings)   # okunamayan bir şey varsa burada söyler

<Learned 11 facts, 4 tables, 3104 evidence via pdf from '#pdf:fw9.pdf'>
süre: 1.4 sn


In [6]:
sorular = [
    "What is the purpose of Form W-9",          # belgede var
    "Which agency issues this form",            # belgede var
    "What is the revision date of this form",   # belgede var
    "How much does this form cost",             # YOK — çekinmeli
    "Who is the current president",             # YOK — çekinmeli
]
for s in sorular:
    t0 = time.time()
    c = belge.ask(s, explain=True)
    isaret = "çekindi" if c.abstained else "cevapladı"
    print(f"> {s}\n  [{isaret}] {c}   ({time.time()-t0:.1f} sn)\n")

> What is the purpose of Form W-9
  [çekindi] person or to certify your TIN when required.   (22.0 sn)

> Which agency issues this form
  [çekindi] I do not know.   (12.5 sn)

> What is the revision date of this form
  [çekindi] I do not know.   (12.2 sn)

> How much does this form cost
  [çekindi] I do not know.   (13.3 sn)

> Who is the current president
  [çekindi] I do not know.   (13.8 sn)



## 5. Kendi sorularını sor

Aşağıdaki listeyi değiştir ve çalıştır. Belgeyi okumak istersen: `print(okundu)` sana kaç cümle
indekslendiğini söylüyordu; içeriği görmek için `belge.session.evidence.sentences[:20]` bakabilirsin.

In [7]:
kendi_sorularim = [
    "What is a TIN",
    "When should I use Form W-9",
]
for s in kendi_sorularim:
    print(f"> {s}\n  {belge.ask(s)}\n")

> What is a TIN
  A TIN is a taxpayer identification number. (~ #pdf:fw9.pdf)



KeyboardInterrupt: 

## 6. Konuşarak öğretmek — yeniden eğitim yok

LLM'de yeni bilgi öğretmek için modeli yeniden eğitmen gerekir. Burada sadece söylüyorsun.

In [ ]:
akil = Memory()
print("önce :", akil.ask("zerbalit nedir"))          # bilmiyor
akil.learn("Zerbalit bir metaldir.")                  # öğret
print("sonra:", akil.ask("zerbalit nedir"))          # biliyor
print("olgu sayısı:", len(akil))

## 7. Belleğin içini görmek

RAG'de "neden bu cevabı verdin" sorusunun karşılığı yoktur. Burada her olgu görülebilir ve kaynağını taşır.

In [ ]:
for olgu in akil.facts[:10]:
    print(olgu)

print("\nbir kavramın çevresi:", akil.about("zerbalit"))

## 8. Kalıcılık — bellek dosyaya yazılır

Bir dosya yolu verirsen bellek diske kaydedilir ve sonraki oturumda kaldığı yerden devam eder.

In [ ]:
kalici = Memory("examples/deneme.lmm")
kalici.learn("Vorlin bir içecektir.")
kalici.save()

yeniden = Memory("examples/deneme.lmm")     # yeni oturum, aynı bellek
print("hatırlıyor mu:", yeniden.ask("vorlin nedir"))

## 9. Excel / CSV — sıfır model çağrısı

Tablolar dil modeline hiç uğramaz: satır = varlık, sütun = alan, hücre = değer olarak doğrudan grafa yazılır.
Bu yüzden milisaniyeler sürer ve hiçbir API maliyeti yoktur.

(Elinde bir xlsx varsa yolunu aşağıya yaz.)

In [ ]:
import csv, pathlib
pathlib.Path("examples/urunler.csv").write_text(
    "urun,kategori,fiyat\n"
    "kahve,icecek,45\n"
    "cay,icecek,30\n"
    "defter,kirtasiye,80\n", encoding="utf-8")

tablo = Memory()
t0 = time.time()
print(tablo.learn("examples/urunler.csv"), f"({(time.time()-t0)*1000:.0f} ms)")

c = tablo.ask("kahve fiyatı nedir", explain=True)
print("cevap:", c, "| graftan mı:", c.from_graph)

## 10. Aynı soruyu iki kez sor — ikincisi bedava

Bellek değişmediyse cevap yeniden üretilmez.

In [ ]:
t0 = time.time(); belge.ask("What is the purpose of Form W-9"); ilk = time.time()-t0
t0 = time.time(); belge.ask("What is the purpose of Form W-9"); ikinci = time.time()-t0
print(f"ilk sorgu: {ilk:.2f} sn\nikinci   : {ikinci:.3f} sn  (önbellek)")

---
## Özet — API'nin tamamı

```python
m = Memory(path=None, cache=True)   # path verirsen kalıcı
m.learn(dosya_veya_metin)           # pdf · docx · xlsx · csv · txt · düz metin
m.ask(soru, explain=False)          # explain=True → .abstained .sources .from_graph
m.save(path=None)
m.about(etiket)                     # bir kavramın çevresindeki olgular
m.facts                             # tüm olgular
len(m)                              # olgu sayısı
```

**Denemeye değer:** `os.environ["LMM_BACKEND"] = "gguf"` yapıp aynı defteri yerel motorla koş —
API maliyeti sıfır olur, cevaplar daha kaba olur. Farkı kendi gözünle görürsün.